<a href="https://colab.research.google.com/github/rhodes-byu/cs-stat-180/blob/main/notebooks/18-ai-nba-stats-gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemini API Tutorial: Ask Questions About NBA Stats

This notebook is a beginner-friendly introduction to using a generative AI API.

By the end, you will know how to:
- create a Gemini client with an API key
- send a simple prompt to the API
- describe a dataset clearly so the model can reason about it
- use tool calling so Gemini writes pandas code, we run it locally, and Gemini explains the result

Workflow:
`question -> Gemini chooses a pandas query -> Python runs the query -> Gemini explains the answer`

**Dataset:** [NBA Players Stats Since 1950](https://www.kaggle.com/datasets/drgilermo/nba-players-stats)

**API key:** [Google AI Studio](https://aistudio.google.com/app/apikey)

**Important note:** this dataset ends in 2017, so newer seasons and players are not included.

## 1. Install the packages

We need:
- `google-genai` for the Gemini API
- `kagglehub` to download the NBA dataset

In [ ]:
# Install the Gemini SDK and the helper package used to download the dataset.
# Remove --quiet if you want to see the full installation log.
!pip install google-genai kagglehub --quiet

print("Packages installed.")

## 2. Download and clean the NBA dataset

Each row in this file represents one player in one NBA season.
Before we ask questions, we clean a few columns so pandas can work with them reliably.

In [ ]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd

DATASET_NAME = "drgilermo/nba-players-stats"
CSV_NAME = "Seasons_Stats.csv"

NUMERIC_COLUMNS = [
    "G", "GS", "MP", "PTS", "TRB", "ORB", "DRB", "AST", "STL", "BLK",
    "TOV", "PF", "FG", "FGA", "FG%", "3P", "3PA", "3P%", "2P", "2PA",
    "2P%", "FT", "FTA", "FT%", "eFG%", "TS%", "PER", "USG%", "WS",
    "WS/48", "OWS", "DWS", "BPM", "VORP", "Age"
]


def load_nba_data(csv_path: Path) -> pd.DataFrame:
    """Load and clean the NBA season stats CSV.

    Args:
        csv_path: Path to the `Seasons_Stats.csv` file downloaded from Kaggle.

    Returns:
        A cleaned pandas DataFrame where `Year` is numeric, player names are
        stripped of trailing `*`, and selected stat columns are converted to
        numeric values when possible.

    Example:
        df = load_nba_data(csv_path)
        df[["Player", "Year", "PTS"]].head()
    """
    data = pd.read_csv(csv_path)

    data["Year"] = pd.to_numeric(data["Year"], errors="coerce")
    data = data.dropna(subset=["Year"]).copy()
    data["Year"] = data["Year"].astype(int)

    # Some player names include a trailing * for All-Star or Hall of Fame markers.
    # Removing it makes player searches easier for beginners.
    data["Player"] = data["Player"].astype(str).str.strip().str.replace("*", "", regex=False)

    for column in NUMERIC_COLUMNS:
        if column in data.columns:
            data[column] = pd.to_numeric(data[column], errors="coerce")

    return data


data_dir = Path(kagglehub.dataset_download(DATASET_NAME))
csv_path = data_dir / CSV_NAME
df = load_nba_data(csv_path)

print(f"Loaded {len(df):,} rows from {csv_path.name}")
print(f"Players: {df['Player'].nunique():,}")
print(f"Seasons: {df['Year'].min()}-{df['Year'].max()}")

df[["Player", "Year", "Tm", "G", "PTS", "AST", "TRB", "BLK", "TS%"]].head()

## 3. Create your Gemini client

Get an API key from [Google AI Studio](https://aistudio.google.com/app/apikey), then paste it below.

In a real project, you would usually store the key in an environment variable instead of typing it into a notebook.

In [ ]:
import getpass

from google import genai
from google.genai import types

API_KEY = getpass.getpass("Paste your Gemini API key: ")
MODEL = "gemini-2.5-flash"

client = genai.Client(api_key=API_KEY)
print(f"Client ready for {MODEL}.")

## 4. Send a very small first prompt

Before we build anything fancy, it helps to see the simplest possible API call.
We send plain text to Gemini and print the plain-text response.

In [ ]:
# This is the smallest useful Gemini request in the notebook.
intro_response = client.models.generate_content(
    model=MODEL,
    contents="In 2 sentences, explain what an API does for a beginner."
)

print(intro_response.text)

## 5. Give the model a clear description of the dataset

Large language models do not automatically understand your pandas DataFrame.
We need to describe the columns, the date range, and important rules.

This step is part of **prompt engineering**.
A good prompt makes better code and better answers.

In [ ]:
from textwrap import dedent

DATA_DICTIONARY = dedent(
    """
    DataFrame name: df
    Rows: about 24,000
    Meaning of each row: one player in one NBA season
    Time span: 1950 through 2017

    Important identity columns
    - Year: end year of the season. Example: 2000 means the 1999-2000 season.
    - Player: full player name.
    - Pos: position such as PG, SG, SF, PF, or C.
    - Tm: team abbreviation. TOT means a combined total row for a player traded mid-season.
    - Age: player age during that season.

    Important counting stats
    - G: games played
    - GS: games started
    - MP: total minutes played
    - PTS: total points
    - TRB: total rebounds
    - AST: total assists
    - STL: total steals, available starting in 1973-74
    - BLK: total blocks, available starting in 1973-74
    - TOV: total turnovers, available starting in 1977-78
    - FG, FGA, 3P, 3PA, 2P, 2PA, FT, FTA are all season totals

    Rule for counting stats
    - To create per-game stats, divide a counting stat by G.
    - Example: BLK / G gives blocks per game.

    Important rate stats that should NOT be divided by G
    - FG%: field goal percentage
    - 3P%: three-point percentage
    - 2P%: two-point percentage
    - FT%: free throw percentage
    - eFG%: effective field goal percentage
    - TS%: true shooting percentage
    - PER: player efficiency rating
    - USG%: usage percentage
    - WS/48: win shares per 48 minutes
    - BPM: box plus/minus
    - VORP: value over replacement player

    Useful query habits
    - For season leaderboards, exclude TOT rows to avoid double-counting traded players.
    - For player searches, use str.contains(..., case=False, na=False).
    - For career per-game leaderboards, create a per-game column first, then group by Player and average it.

    Dataset warnings
    - Post-2017 players are not in this file.
    - Some columns are NaN before the stat started being recorded.
    """
).strip()

print(f"Data dictionary ready ({len(DATA_DICTIONARY.split())} words).")

## 6. Build the prompt and the local tool

Here is the key idea in this notebook:

1. We ask Gemini an NBA question.
2. Gemini must call a tool named `run_pandas_query`.
3. The tool contains short pandas code.
4. Python runs that code on `df`.
5. Gemini explains the result in plain English.

This pattern is more reliable than asking the model to answer from memory, because the answer is tied to the actual dataset.

In [ ]:
import traceback

QUESTION_RULES = dedent(
    """
    You are helping a beginner explore NBA stats with pandas.

    Use the `run_pandas_query` tool exactly once before answering the question.
    The tool code must use only df, pd, and np.
    Assign the final answer object to a variable named result.

    Rules for the tool code:
    - Write 1 to 6 lines of executable Python.
    - Use counting_stat / G for per-game rates.
    - Do not divide percentage stats or advanced rate stats by G.
    - Exclude TOT rows for season leaderboards.
    - If the tool returns an error, explain the error instead of guessing.
    - Final answer style: conversational, specific, and short.
    """
).strip()


def build_question_prompt(question: str) -> str:
    """Build the full prompt sent to Gemini for an NBA question.

    Args:
        question: The natural-language question the student wants to ask about
            the NBA dataset.

    Returns:
        A single prompt string containing the tool rules, the data dictionary,
        and the student's question.

    Example:
        prompt = build_question_prompt("Who led the league in assists in 1990?")
        print(prompt[:400])
    """
    return dedent(
        f"""
        {QUESTION_RULES}

        Dataset description:
        {DATA_DICTIONARY}

        Student question: {question}
        """
    ).strip()


def show_prompt(question: str) -> None:
    """Print the exact prompt that will be sent to Gemini.

    Args:
        question: The question you plan to pass to `ask(...)`.

    Returns:
        None. This function prints text so you can inspect the prompt.

    Example:
        show_prompt("Which guards had the highest career steals per game?")
    """
    print(build_question_prompt(question))


QUERY_FUNCTION = types.FunctionDeclaration(
    name="run_pandas_query",
    description="Run short Python/pandas code against df and return a compact preview of the result.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": (
                    "Executable Python that uses only df, pd, and np. "
                    "Assign the final object to a variable named result."
                ),
            }
        },
        "required": ["code"],
    },
)

QUERY_TOOL = types.Tool(function_declarations=[QUERY_FUNCTION])


def clean_model_code(code: str) -> str:
    """Remove Markdown code fences from model-generated Python.

    Args:
        code: Raw text returned by Gemini. Sometimes the model wraps Python in
            triple backticks before sending it through the tool.

    Returns:
        A cleaned Python string that can be passed to `exec()`.

    Example:
        clean_model_code("```python\nresult = df.head()\n```")
    """
    code = code.strip()
    if code.startswith("```"):
        lines = code.splitlines()
        code = "\n".join(lines[1:])
        if code.endswith("```"):
            code = "\n".join(code.splitlines()[:-1])
    return code.strip()


def format_result_for_model(result) -> str:
    """Convert a pandas result into a short text preview for Gemini.

    Args:
        result: Usually a pandas DataFrame, pandas Series, or a scalar value
            produced by model-written pandas code.

    Returns:
        A compact text summary that is short enough to send back to Gemini in a
        follow-up API call.

    Example:
        sample = df[["Player", "PTS"]].head(3)
        print(format_result_for_model(sample))
    """
    if isinstance(result, pd.DataFrame):
        preview = result.head(20).to_string(index=False)
        return f"Result ({len(result)} rows):\n{preview}"

    if isinstance(result, pd.Series):
        preview = result.head(20).to_string()
        return f"Result:\n{preview}"

    return f"Result: {result}"


def run_pandas_query(code: str, debug: bool = False) -> dict:
    """Run model-generated pandas code against the local DataFrame.

    Args:
        code: Short Python code that uses only `df`, `pd`, and `np`, and stores
            the final answer in a variable named `result`.
        debug: If True, print the generated code and any Python traceback.

    Returns:
        A dictionary with `status`, `code`, and `data_context` fields. This is
        the payload sent back to Gemini after the tool executes.

    Example:
        run_pandas_query(
            "result = df[df['Player'].str.contains('Jordan', case=False, na=False)].head()",
            debug=True,
        )

    Warning:
        This classroom demo uses `exec()`, which can run arbitrary Python code.
        That is convenient for teaching, but unsafe in production.
    """
    code = clean_model_code(code)
    local_vars = {"df": df, "pd": pd, "np": np}

    if debug:
        print("-- Generated pandas code --")
        print(code)
        print()

    try:
        exec(code, {}, local_vars)
        result = local_vars.get("result")

        if result is None:
            return {
                "status": "error",
                "code": code,
                "data_context": "The code ran, but it did not create a variable named `result`.",
            }

        return {
            "status": "ok",
            "code": code,
            "data_context": format_result_for_model(result),
        }

    except Exception as error:
        if debug:
            traceback.print_exc()

        return {
            "status": "error",
            "code": code,
            "data_context": f"Code failed with error: {error}",
        }


FIRST_PASS_CONFIG = types.GenerateContentConfig(
    tools=[QUERY_TOOL],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(
            mode="ANY",
            allowed_function_names=["run_pandas_query"],
        )
    ),
    temperature=0,
)

FINAL_PASS_CONFIG = types.GenerateContentConfig(
    tools=[QUERY_TOOL],
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(mode="NONE")
    ),
    temperature=0.3,
)


def ask(question: str, debug: bool = False) -> None:
    """Ask Gemini a question about the NBA dataset.

    Args:
        question: A natural-language question such as "Who were the top scorers
            in 1988?" or "Compare Duncan and Garnett as rebounders."
        debug: If True, print the pandas code Gemini generated before the final
            explanation is shown.

    Returns:
        None. The function prints Gemini's final explanation to the notebook.

    Example:
        ask("Which centers had the highest career blocks per game?", debug=True)
        ask("Who had the best true shooting percentage with at least 5 seasons?")
    """
    prompt = build_question_prompt(question)
    conversation = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=prompt)],
        )
    ]

    first_response = client.models.generate_content(
        model=MODEL,
        contents=conversation,
        config=FIRST_PASS_CONFIG,
    )

    if not first_response.function_calls:
        print(first_response.text)
        return

    tool_call = first_response.function_calls[0]
    tool_code = tool_call.args.get("code", "")
    tool_result = run_pandas_query(tool_code, debug=debug)

    conversation.append(first_response.candidates[0].content)
    conversation.append(
        types.Content(
            role="tool",
            parts=[
                types.Part.from_function_response(
                    name=tool_call.name,
                    response=tool_result,
                )
            ],
        )
    )

    final_response = client.models.generate_content(
        model=MODEL,
        contents=conversation,
        config=FINAL_PASS_CONFIG,
    )

    print(final_response.text)


print("Ready. Try ask('Which centers had the highest career blocks per game?')")


## 7. Function reference

The helper functions now include longer docstrings with `Args`, `Returns`, and examples.

Useful ones to inspect:
- `help(load_nba_data)`
- `help(build_question_prompt)`
- `help(show_prompt)`
- `help(run_pandas_query)`
- `help(ask)`

Quick usage examples:
- `show_prompt("Who led the league in assists in 1990?")`
- `ask("Compare Tim Duncan and Kevin Garnett as rebounders.", debug=True)`

Reading function documentation is a normal part of learning APIs, so treat these helpers like mini API endpoints inside the notebook.

In [ ]:
show_prompt("Which centers had the highest career blocks per game?")

## 8. Ask questions about the NBA data

These are example prompts you can run right away.
Set `debug=True` when you want to see the pandas code Gemini generated for the tool.

In [ ]:
ask("How did Michael Jordan perform over his career?")

In [ ]:
ask("Who were the top scorers in the 1988 season?")

In [ ]:
ask("Compare LeBron James and Kobe Bryant across points, assists, and rebounds per game.")

In [ ]:
ask("Which centers had the highest career blocks per game?", debug=True)

In [ ]:
ask("Who had the best true shooting percentage with at least 5 seasons?")

## 9. Try your own prompts

Here are a few question ideas:
- `Who led the league in assists in 1990?`
- `What were Shaquille O'Neal's best scoring seasons?`
- `Compare Tim Duncan and Kevin Garnett as rebounders.`
- `Which guards had the highest career steals per game?`

Try one below.

In [ ]:
ask("YOUR QUESTION HERE", debug=True)

## 10. Reflection questions

1. Where in this notebook do we actually make an API request?
2. Which parts of this notebook count as prompt engineering?
3. Why is tool calling better than asking Gemini to answer from memory?
4. What did `debug=True` reveal about the model's pandas code?
5. Why is `exec()` acceptable for a classroom demo but dangerous in production?
6. How could you make this project safer in a real application?